# FedOPF: HFL approach based on APPFL


In [1]:
import argparse
from omegaconf import OmegaConf
from appfl.agent import ClientAgent, ServerAgent

import time

[W112 10:20:06.833117467 Context.cpp:281] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


In [2]:
cu_config_path = "./resources/configs/fedopf_central_unit_hfl.yaml"
server_config_path = "./resources/configs/fedopf_server_hfl.yaml"
client_config_path = "./resources/configs/fedopf_client_1.yaml"

client_ids = [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]
# client_ids = [14, 250, 118, 89, 1354, 1664, 197, 60, 73, 57, 162, 793]
num_clients = len(client_ids)

In [3]:
central_unit_config = OmegaConf.load(cu_config_path) # Load central unit configs for HFL

# Load server agent config and set corresponding fields for # of clusters (CFL_GP)
n_models = central_unit_config.cu_configs.n_models
server_agent_configs = [
    OmegaConf.load(server_config_path) for _ in range(n_models)
]
# server_agent_config.server_configs.num_clients = num_clients
# Create server agent (CFL_GP)
server_agents = [ServerAgent(server_agent_config=server_agent_configs[i]) for i in range(n_models)]

# Create global agent (for foundation model <== HFL)
global_agent_config = OmegaConf.load(server_config_path)
global_agent = ServerAgent(server_agent_config=global_agent_config)

appfl: ✅[2026-01-12 05:12:15,643 server]: Logging to ./output/result_Server_2026-01-12-05-12-15.txt


In [4]:
# Load base client configurations and set corresponding fields for different clients
client_agent_configs = [
    OmegaConf.load(client_config_path) for _ in range(num_clients)
]

for i, id in enumerate(client_ids):
    client_agent_configs[i].client_id = f"Client{i+1}"
    # client_agent_configs[i].data_configs.dataset_kwargs.num_clients = num_clients
    client_agent_configs[i].data_configs.dataset_kwargs.client_id = id
    # client_agent_configs[i].data_configs.dataset_kwargs.visualization = (
    #     True if i == 0 else False
    # )
    
    # only enable wandb for the first client is sufficient for logging all clients in serial run
    if hasattr(client_agent_configs[i], "wandb_configs") and client_agent_configs[i].wandb_configs.get("enable_wandb", False):
        if i == 0:
            client_agent_configs[i].wandb_configs.enable_wandb = True
        else:
            client_agent_configs[i].wandb_configs.enable_wandb = False

In [5]:
# Load client agents
client_agents = [
    ClientAgent(client_agent_config=client_agent_configs[i]) for i in range(num_clients)
]

appfl: ✅[2026-01-12 05:12:15,896 Client1]: Logging to ./output/result_Client1_2026-01-12-05-12-15.txt
/home/super/data1/skj/FedOPF-APPFL/HFL/resources/dataset/acopf_prob_advanced.py:134: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],
appfl: ✅[2026-01-12 05:12:17,020 Client2]: Logging to ./output/result_Client2_2026-01-12-05-12-17.txt
INFO:appfl.logger.client_logger_Client2:Logging to ./output/result_Client2_2026-01-12-05-12-17.txt
appfl: ✅[2026-01-12 05:12:17,441 Client3]: Logging to ./output/result_Client3_2026-01-12-05-12-17.txt
INFO:appfl.logger.client_logger_Client3:Logging to ./output/result_Client3_2026-01-12-05-12-17.txt
appfl: ✅[2026-01-12 05:12:17,823 Client4]: Logging to ./output/resu

In [6]:
# Get additional client configurations from the server
client_config_from_server = server_agents[0].get_client_configs()
# print(OmegaConf.to_yaml(server_agent.get_client_configs()))
client_config_from_server.model_configs.model_name = 'GraphOPF'
client_config_from_server.model_configs.model_path = "./resources/model/graphopf_client.py"

for client_agent in client_agents:
    client_config_from_server.model_configs.model_kwargs.client_id = client_agent.dataset.nbus
    client_agent.load_config(client_config_from_server)

In [7]:
central_unit_config.cu_configs

{'gg_correction_period': 1, 'n_models': 4, 'clustering_period': 2, 'clustering_termination_threshold': 50, 'cu_compressor_configs': {'use_reduced_G': False, 'gradient_compression_ratio': 100}, 'num_comm_rounds': 200, 'unknown_k': False, 'warmup_epoch': 0, 'checkpoints_interval': 100}

In [8]:
import torch
import numpy as np 
from utils.cfl_gp import get_num_cluster, spectral_clustering_and_matching
from collections import OrderedDict # NOTE: kj
import copy

def get_mdl_params(model_params):
    n_par=0
    for name, param in model_params:
        n_par += len(param.data.reshape(-1))
    param_mat = np.zeros(n_par).astype('float32')
    idx = 0
    for name, param in model_params:
        temp = param.data.cpu().numpy().reshape(-1)
        param_mat[idx:idx + len(temp)] = temp
        idx += len(temp)
    return np.copy(param_mat)

# Main code: run HFL
# HFL parameters
clustering_period = central_unit_config.cu_configs.clustering_period
gg_correction_period = central_unit_config.cu_configs.gg_correction_period
use_reduced_G = central_unit_config.cu_configs.cu_compressor_configs.use_reduced_G
gradient_compression_ratio = central_unit_config.cu_configs.cu_compressor_configs.gradient_compression_ratio
clustering_termination_threshold = central_unit_config.cu_configs.clustering_termination_threshold
estimated_cluster_ids = None
estimated_cluster_ids_new = None
unknown_k = central_unit_config.cu_configs.unknown_k
warmup_epoch = central_unit_config.cu_configs.warmup_epoch
save_checkpoint_interval = central_unit_config.cu_configs.checkpoints_interval

n_params_per_model = sum(param.numel() for param in server_agents[0].model.parameters())
n_params_compressed_gradient = n_params_per_model
if use_reduced_G:
    n_params_compressed_gradient = n_params_per_model // gradient_compression_ratio
    random_G_indices = np.random.choice(np.arange(n_params_per_model), size=n_params_compressed_gradient, replace=False)
print("n_params_per_model: ", n_params_per_model)
print("n_params_compressed_gradient: ", n_params_compressed_gradient)

gradient_profile_matrix = np.zeros(shape=(n_models * n_params_compressed_gradient, num_clients))
criterion_model_index = 0
CLUSTERING_CONVERGENCE = False

# group-global corr. update
group_global_state_param_list = np.zeros((n_models, n_params_compressed_gradient)) # .astype('float32')        

metrics = {} # data tracking

for round in range(central_unit_config.cu_configs.num_comm_rounds):
    # Model save
    if round == central_unit_config.cu_configs.num_comm_rounds-1:
        for client_agent in client_agents:
            client_agent.save_checkpoint()
        for m in range(n_models):
            server_agents[m].save_checkpoint(server_id=m)
        global_agent.save_checkpoint(server_id=-1)

    if round != 0:
        # Load the new global model from the server
        for cluster_idx, client_cluster in enumerate(client_clusters):
            print("load new group model")
            for client, new_global_model_future in zip(client_cluster, group_global_models[cluster_idx]):
                client.load_parameters(new_global_model_future.result())

    cr_start_time = time.time()
    for c_idx, client_agent in enumerate(client_agents):
        # - Select models for downlink transmission
        if round % clustering_period == 0 and CLUSTERING_CONVERGENCE is not True:
            if round == 0:
                query_model_indicies = [0]
            elif estimated_cluster_ids[c_idx] == criterion_model_index:
                query_model_indicies = [estimated_cluster_ids[c_idx]]
            else:
                query_model_indicies = [criterion_model_index, estimated_cluster_ids[c_idx]]
        else:
            query_model_indicies = [estimated_cluster_ids[c_idx]]
        
        # - local update or gradient calculation
        for model_idx in query_model_indicies:
            if round == 0:
                estimated_cluster_id = 0
                # Load initial global model from the server
                init_global_model = server_agents[estimated_cluster_id].get_parameters(serial_run=True)
                client_agent.load_parameters(init_global_model)
            else:
                estimated_cluster_id = estimated_cluster_ids[c_idx]  
                print(server_agents[estimated_cluster_id].get_parameters(serial_run=True)['layers.0.edge_aggr.0.weight'][:2,:])
            
            group_global_state_param_list_ = torch.tensor(group_global_state_param_list[estimated_cluster_id], dtype=torch.float32)        
            ## client local training
            client_agent.train(round=round, gg_corr_params=group_global_state_param_list_)
            local_model = client_agent.get_parameters()
            if isinstance(local_model, tuple):
                local_model, metadata = local_model[0], local_model[1]
            else:
                metadata = {}

            ## NOTE: kj
            local_model = OrderedDict((k,v) for k,v in local_model.items() if k.startswith("layers"))
            if round % clustering_period == 0 and model_idx == criterion_model_index and CLUSTERING_CONVERGENCE is not True:
                # vectorized_model_info = flatten_tensor(local_model).clone().cpu().detach().numpy() ~~~ # <== shared NNs 이 들어가야 함! "local_model" 자체가 애초에 shared layers를 불러오게끔 세팅되어 있음.
                vectorized_model_info = torch.cat([value.flatten() for value in local_model.values()])
                vectorized_model_info = vectorized_model_info.clone().cpu().detach().numpy()
                if use_reduced_G is True:
                    selected_vectorized_model_info = vectorized_model_info[random_G_indices]
                    vectorized_model_info = selected_vectorized_model_info

                # Cumulative Averaging
                beta = (1 / (np.floor((round + 1) / (n_models * clustering_period)) + 1))
                # global_logger.info("beta:{}".format(beta))
                gradient_profile_matrix[(criterion_model_index)*n_params_compressed_gradient : (criterion_model_index + 1)*n_params_compressed_gradient,c_idx] = \
                    gradient_profile_matrix[(criterion_model_index)*n_params_compressed_gradient : (criterion_model_index + 1)*n_params_compressed_gradient,c_idx] * \
                        (1 - beta) + (beta) * vectorized_model_info

    # - Clustering & Matching
    if unknown_k is True and round < warmup_epoch:
        proposed_k = get_num_cluster(gradient_profile_matrix, n_centers=n_models,
                                        n_clients=num_clients,
                                        estimated_cluster_ids_old=estimated_cluster_ids)
        estimated_cluster_ids = np.zeros(shape=num_clients, dtype=int)
        estimated_cluster_ids_new = np.zeros(shape=num_clients, dtype=int)
        consistency_cnt = 0
    elif unknown_k is True and round == warmup_epoch:
        n_models = get_num_cluster(gradient_profile_matrix, n_centers=n_models,
                                        n_clients=num_clients,
                                        estimated_cluster_ids_old=estimated_cluster_ids)
        group_global_models = [[] for _ in range(n_models)]
        print("Set number of clusters as {}.".format(n_models))

        for m in np.arange(1, n_models):
            # copy_weight2(target=self.models[m], source=self.models[0])
            server_agents[m].model.load_state_dict(server_agents[0].get_parameters(serial_run=True))
    
        estimated_cluster_ids = np.zeros(shape=num_clients, dtype=int)
        estimated_cluster_ids_new = np.zeros(shape=num_clients, dtype=int)
        consistency_cnt = 0
    elif round % clustering_period == 0 and round < clustering_termination_threshold:
        print("sepctral_clustering_and_matching")
        # print(gradient_profile_matrix.shape)
        criterion_model_index = (criterion_model_index + 1) % n_models
        info = spectral_clustering_and_matching(gradient_profile_matrix, n_centers=n_models,
                                                n_clients=num_clients,
                                                estimated_cluster_ids_old=estimated_cluster_ids,
                                                clustering_algorithm='KMeans') # KMeans, DBSCAN, AgglomerativeClustering
        estimated_cluster_ids_new = info["estimated_cluster_ids"]  # update cluster ids.
        reduced_gradient_profile_matrix = info["reduced_gradient_profile_matrix"]
        singular_values = info["singular_values"]

        estimated_cluster_ids = estimated_cluster_ids_new  # np.zeros(shape=self.n_clients, dtype=int)        

    print("cluster ids: {}".format(estimated_cluster_ids))

    # - Model update
    group_global_models = [[] for _ in range(n_models)]
    clustered_client_indices = [np.where(estimated_cluster_ids == cluster_id)[0] for cluster_id in range(n_models)]
    client_clusters = [[client_agents[i] for i in client_indices] for client_indices in clustered_client_indices]
    for cluster_idx, client_cluster in enumerate(client_clusters):
        if len(client_cluster) >= 1:
            # print("group model update!")
            server_agents[cluster_idx].num_clients = len(client_cluster)
            # server_agents[cluster_idx]._set_num_clients()

            # TODO: Check handling server configs for clustered FL. 
            server_agents[cluster_idx].aggregator.aggregator_configs.num_clients = len(client_cluster)
            server_agents[cluster_idx].scheduler.num_clients = len(client_cluster)
            # server_agents[cluster_idx]._load_scheduler()
            # server_agents[cluster_idx].scheduler.aggregation_kwargs['num_clients'] = len(client_cluster)

            for client in client_cluster:
                c_model = client.get_parameters()
                if isinstance(c_model, tuple):
                    c_model, metadata = c_model[0], c_model[1]
                else:
                    metadata = {}
                ## NOTE: kj
                c_model = OrderedDict((k,v) for k,v in c_model.items() if k.startswith("layers"))
                # "Send" local model to server and get a Future object for the new global model
                # The Future object will be resolved when the server receives local models from all clients
                new_global_model_future = server_agents[cluster_idx].global_update(
                    client_id=client.get_id(),
                    local_model=c_model,
                    blocking=False, # False
                    **metadata,
                )
                group_global_models[cluster_idx].append(new_global_model_future)

    # Group-Global Correction
    if round % gg_correction_period == 0:
        print("Update Group model and GG-Correction.")
        global_models = []
        for client_agent in client_agents:
            local_model_gg = client_agent.get_parameters()
            if isinstance(local_model_gg, tuple):
                local_model_gg, metadata = local_model_gg[0], local_model_gg[1]
            else:
                metadata = {}
            ## NOTE: kj
            local_model_gg = OrderedDict((k,v) for k,v in local_model_gg.items() if k.startswith("layers"))
            # "Send" local model to server and get a Future object for the new global model
            # The Future object will be resolved when the server receives local models from all clients
            new_global_model = global_agent.global_update(
                client_id=client_agent.get_id(),
                local_model=local_model_gg,
                blocking=False,
                **metadata,
            )
            global_models.append(new_global_model)

        beta_ = 0.9
        for m in range(n_models):
            # group_models_params = get_mdl_params(server_agents[m].get_parameters(init_model=False).items())
            # global_model_params = get_mdl_params(global_agent.get_parameters(init_model=False).items())
            group_models_params = get_mdl_params(server_agents[m].get_parameters(serial_run=True).items())
            global_model_params = get_mdl_params(global_agent.get_parameters(serial_run=True).items())
            group_global_state_param_list[m,:] = (1-beta_)*group_global_state_param_list[m,:] + beta_*(group_models_params - global_model_params)

    time_for_communication_round = time.time() - cr_start_time
    # ====================================
    # Data Tracking and save Results
    # ====================================
    data_tracking = {
        "estimated_cluster_ids": copy.deepcopy(estimated_cluster_ids),
        "reduced_gradient_profile_matrix": copy.deepcopy(reduced_gradient_profile_matrix),
        # "singular_values": copy.deepcopy(self.singular_values),
        "time_for_communication_round": time_for_communication_round
    }
    metrics["c_round_" + str(round)] = data_tracking
    if round == central_unit_config.cu_configs.num_comm_rounds-1: 
        global_agent.save_data_tracking(metrics)

n_params_per_model:  973050
n_params_compressed_gradient:  973050


appfl: ✅[2026-01-12 05:12:39,178 Client1]:      Round      Epoch       Time Train Loss Train Accuracy
INFO:appfl.logger.client_logger_Client1:     Round      Epoch       Time Train Loss Train Accuracy
appfl: ✅[2026-01-12 05:12:40,524 Client1]:          0          0     1.3298     0.8421           72.4
INFO:appfl.logger.client_logger_Client1:         0          0     1.3298     0.8421           72.4
appfl: ✅[2026-01-12 05:12:41,196 Client1]:          0          1     0.6699     0.2932           91.2
INFO:appfl.logger.client_logger_Client1:         0          1     0.6699     0.2932           91.2
appfl: ✅[2026-01-12 05:12:41,810 Client1]:          0          2     0.6116     0.2484           74.4
INFO:appfl.logger.client_logger_Client1:         0          2     0.6116     0.2484           74.4
appfl: ✅[2026-01-12 05:12:42,438 Client1]:          0          3     0.6249     0.2620           81.2
INFO:appfl.logger.client_logger_Client1:         0          3     0.6249     0.2620           

sepctral_clustering_and_matching
cluster ids: [0 0 2 0 0 0 0 3 0 1 1 0]
Update Group model and GG-Correction.
load new group model
load new group model
load new group model
load new group model
tensor([[ 0.2684,  0.2911, -0.0850,  0.3216, -0.0761,  0.0742, -0.1706,  0.2045],
        [ 0.3077, -0.2608,  0.3003,  0.0591,  0.2564,  0.0450,  0.1695, -0.0450]])


appfl: ✅[2026-01-12 05:14:39,428 Client1]:          1          0     0.5728     0.3961           77.2
INFO:appfl.logger.client_logger_Client1:         1          0     0.5728     0.3961           77.2
appfl: ✅[2026-01-12 05:14:39,995 Client1]:          1          1     0.5652     0.2506           92.8
INFO:appfl.logger.client_logger_Client1:         1          1     0.5652     0.2506           92.8
appfl: ✅[2026-01-12 05:14:40,572 Client1]:          1          2     0.5740     0.2371           94.0
INFO:appfl.logger.client_logger_Client1:         1          2     0.5740     0.2371           94.0
appfl: ✅[2026-01-12 05:14:41,139 Client1]:          1          3     0.5646     0.2331           93.6
INFO:appfl.logger.client_logger_Client1:         1          3     0.5646     0.2331           93.6
appfl: ✅[2026-01-12 05:14:41,709 Client1]:          1          4     0.5682     0.2308           96.8
INFO:appfl.logger.client_logger_Client1:         1          4     0.5682     0.2308           

tensor([[ 0.2684,  0.2911, -0.0850,  0.3216, -0.0761,  0.0742, -0.1706,  0.2045],
        [ 0.3077, -0.2608,  0.3003,  0.0591,  0.2564,  0.0450,  0.1695, -0.0450]])


...

appfl: ✅[2026-01-12 05:18:10,170 Client10]:          2          0     1.6529    43.6268       94.51685
INFO:appfl.logger.client_logger_Client10:         2          0     1.6529    43.6268       94.51685
appfl: ✅[2026-01-12 05:18:11,574 Client10]:          2          1     1.4030    35.2383       95.88765
INFO:appfl.logger.client_logger_Client10:         2          1     1.4030    35.2383       95.88765
appfl: ✅[2026-01-12 05:18:12,940 Client10]:          2          2     1.3648    31.9139        96.1573
INFO:appfl.logger.client_logger_Client10:         2          2     1.3648    31.9139        96.1573
appfl: ✅[2026-01-12 05:18:14,299 Client10]:          2          3     1.3576    31.8051       95.41574
INFO:appfl.logger.client_logger_Client10:         2          3     1.3576    31.8051       95.41574
appfl: ✅[2026-01-12 05:18:15,677 Client10]:          2          4     1.3766    31.3694       97.34832
INFO:appfl.logger.client_logger_Client10:         2          4     1.3766    31.3694 

tensor([[ 0.2654,  0.2878, -0.0877,  0.3147, -0.0718,  0.0756, -0.1464,  0.1970],
        [ 0.3158, -0.2521,  0.3006,  0.0621,  0.2562,  0.0430,  0.1440, -0.0430]])


appfl: ✅[2026-01-12 05:18:21,456 Client11]:          2          0     3.8332   201.3453       85.07693
INFO:appfl.logger.client_logger_Client11:         2          0     3.8332   201.3453       85.07693
appfl: ✅[2026-01-12 05:18:24,340 Client11]:          2          1     2.8826   153.3899      87.446144
INFO:appfl.logger.client_logger_Client11:         2          1     2.8826   153.3899      87.446144
appfl: ✅[2026-01-12 05:18:27,294 Client11]:          2          2     2.9525   146.9007       91.43076
INFO:appfl.logger.client_logger_Client11:         2          2     2.9525   146.9007       91.43076
appfl: ✅[2026-01-12 05:18:30,265 Client11]:          2          3     2.9685   145.8831       87.08461
INFO:appfl.logger.client_logger_Client11:         2          3     2.9685   145.8831       87.08461
appfl: ✅[2026-01-12 05:18:33,051 Client11]:          2          4     2.7841   145.3546       89.96923
INFO:appfl.logger.client_logger_Client11:         2          4     2.7841   145.3546 

tensor([[ 0.2164,  0.2464, -0.0800,  0.3075, -0.0370,  0.0319, -0.1290,  0.1434],
        [ 0.2700, -0.2530,  0.2661,  0.0371,  0.2065,  0.0365,  0.1103, -0.0284]])


cuDSSError: ALLOC_FAILED (2)

In [ ]:
estimated_cluster_ids

array([2, 3, 0, 2, 2, 3, 0, 2, 3, 1, 1, 3])

## Evaluate model on "test dataset" for each client

In [11]:
# Modifies stats in place
def dict_agg(stats, key, value, op='concat'):
    if key in stats.keys():
        if op == 'sum':
            stats[key] += value
        elif op == 'concat':
            stats[key] = np.concatenate((stats[key], value), axis=0)
        else:
            raise NotImplementedError
    else:
        stats[key] = value

In [12]:
from torch_geometric.loader import DataLoader
from pypower.api import makeYbus
import torch
import numpy as np
import time

for client_agent in client_agents:
    print(client_agent.get_id())
    test_dataloader = DataLoader(
            client_agent.test_dataset,
            batch_size=1,
            shuffle=False,
            drop_last=True
        )

    Ybus, Yf, Yt = makeYbus(client_agent.dataset.baseMVA, client_agent.dataset.ppc['bus'], client_agent.dataset.ppc['branch'])
    # branch thermal limit information
    flow_max = (client_agent.dataset.ppc['branch'][:, 5] / client_agent.dataset.baseMVA)**2
    flow_max[flow_max == 0] = np.inf # np.Inf
    flow_max = torch.tensor(flow_max, dtype=torch.float32).to(client_agent.dataset.device)

    test_len = 200
    node_means, node_stds, edge_means, edge_stds = client_agent.dataset.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
    n_means = node_means.to(client_agent.dataset.device)
    n_stds = node_stds.to(client_agent.dataset.device)
    e_means = edge_means.to(client_agent.dataset.device)
    e_stds = edge_stds.to(client_agent.dataset.device)

    client_agent.model.eval()
    test_stats = {}
    test_eps_converge = 1e-4

    # LagM = torch.ones(1, 2*ng + 2*nbus + 2*nl).to(DEVICE) # shape: (1, num_inequalities)
    LagM_sp_g = torch.ones(1, 2).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_q_g = torch.ones(1, 2*client_agent.dataset.ng).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_v_m = torch.ones(1, 2*client_agent.dataset.nbus).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_line_l = torch.ones(1, 2*client_agent.dataset.nl).to(client_agent.dataset.device) # shape: (1, num_inequalities)

    solve_time = []
    with torch.no_grad():
        for (i, Xtest) in enumerate(test_dataloader):
            Xtest = Xtest.to(client_agent.dataset.device)

            start_time = time.time()
            Y = client_agent.model(Xtest, n_means, n_stds, e_means, e_stds)
            end_time = time.time()

            solve_time += [end_time - start_time]

            ## line thermal limit
            pg, qg, vm, va = client_agent.dataset.get_yvars(Y)
            vr = vm*torch.cos(va)
            vi = vm*torch.sin(va)
            vz = torch.complex(vr, vi) # complex voltage

            # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
            If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T
            It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T

            # Calculate the apparent power S
            Sf = vz[:,client_agent.dataset.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
            St = vz[:,client_agent.dataset.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
            Sff = Sf * torch.conj(Sf)
            Stt = St * torch.conj(St)

            # calculate the line thermal limit constraints violation
            diff_Sf = Sff.real - flow_max
            diff_St = Stt.real - flow_max
            # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

            line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
            line_limit_vio_St = torch.clamp(diff_St, 0)
            ###########################################

            # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
            test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = client_agent.loss_fn.forward(client_agent.dataset, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

            dict_agg(test_stats, 'time', np.array(end_time - start_time).reshape(1,-1))

            # test_ineq_p_g = torch.cat([pg - client_agent.dataset.pmax.to(client_agent.dataset.device), client_agent.dataset.pmin.to(client_agent.dataset.device) - pg], dim=1)
            # test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(client_agent.dataset.device)
            test_ineq_p_g = test_ineq_dist[:,:2]
            test_ineq_q_g = test_ineq_dist[:,2:2+2*client_agent.dataset.ng]
            test_ineq_v_m = test_ineq_dist[:,2+2*client_agent.dataset.ng:2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus]
            test_ineq_line_l = test_ineq_dist[:,2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus:]

            dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
            # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

            dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

            pg_rate_torch = (((pg <= client_agent.dataset.pmax.to(client_agent.dataset.device)) & (pg >= client_agent.dataset.pmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            qg_rate_torch = (((qg <= client_agent.dataset.qmax.to(client_agent.dataset.device)) & (qg >= client_agent.dataset.qmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

            v_rate_torch = (((vm <= client_agent.dataset.vmax.to(client_agent.dataset.device)) & (vm >= client_agent.dataset.vmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.nbus)*100
            dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

            sff_rate_torch = ((Sff.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            stt_rate_torch = ((Stt.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

            test_eq_real = test_eq_resid[:,:client_agent.dataset.nbus]
            test_eq_react = test_eq_resid[:,client_agent.dataset.nbus:]
            dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:client_agent.dataset.nbus] <= 1e-2) & (test_eq_resid[:,:client_agent.dataset.nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:client_agent.dataset.nbus].shape[1]*100).detach().cpu().numpy())
            dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,client_agent.dataset.nbus:] <= 1e-2) & (test_eq_resid[:,client_agent.dataset.nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,client_agent.dataset.nbus:].shape[1]*100).detach().cpu().numpy())

    print("GraphLDE obj. value for test samples: ", np.round(np.mean(test_stats['test_obj_cost'])*10000, 4))
    print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
    print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
    print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
    print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
    print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
    print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

    print("\n")
    print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
    print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
    print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
    print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
    print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
    print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
    print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
    print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
    print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
    print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

    print("\n")
    print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
    print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
    print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
    print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
    print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
    print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
    print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

    print("\n")
    print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(test_stats['time']*1e3)))
    print('\n')

# [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]

Client1
GraphLDE obj. value for test samples:  2179.1168
GraphLDE eq. mean for test samples:  4.5920024e-05
GraphLDE eq. max for test samples:  0.00040408384
GraphLDE eq. active mean for test samples:  4.0165687e-05
GraphLDE eq. active max for test samples:  0.00018686285
GraphLDE eq. reactive mean for test samples:  5.167436e-05
GraphLDE eq. reactive max for test samples:  0.000404082


GraphLDE ineq. mean for test samples:  1.755544e-05
GraphLDE ineq. max for test samples:  0.0012682964
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.00014044352
GraphLDE ineq. q_g max for test samples:  0.0012682964
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisfication rate for test samples:  100.0
GraphLDE q_g satisfication rate for test samples: 